# Finding Redundant 1D Subspaces in LLM Activations

This notebook discovers **redundant 1-dimensional subspaces** in language model activations: directions where:
1. Activations have **large projections** (model actively uses this direction)
2. **Removing** this projection has **minimal impact** on output token distributions

## Why Use a Diverse Dataset?

We use the **Anthropic HH-RLHF** dataset (5k diverse prompts) instead of domain-specific data because:

- **Generalization**: Redundancy found across diverse conversations is more robust and fundamental to the model architecture
- **Avoiding Overfitting**: Domain-specific datasets (e.g., harmful content) might find spurious redundancy specific to that domain
- **True Architectural Redundancy**: Directions that remain redundant across questions about science, coding, philosophy, advice, etc. represent genuine architectural inefficiency
- **Better Validation**: If a direction is redundant for diverse topics, it's truly safe to ablate

This approach finds **universally redundant** directions rather than task-specific ones.


In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

# change dir to ..
os.chdir(module_path)

In [2]:
from src.adv_model import AdvModel
import torch
from scripts.utils.load_model import load_model

torch.set_float32_matmul_precision("high")

model, tokenizer = load_model("meta-llama/Llama-3.2-3B-Instruct", torch_dtype="bfloat16")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
INFO 12-18 05:26:59 [src.utils.huggingface:66] Set pad_token to eos_token (id: 128009)
INFO 12-18 05:26:59 [src.utils.huggingface:66] Set pad_token to eos_token (id: 128009)


In [3]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((3072,), eps=1e-05)
    (rotary_emb

In [4]:
from src.univ_attacks.iris import RefusalConfig

refusal_config = RefusalConfig.load("/home/fre.gilad/source/llm-iml/directions/Llama-3.2-3B-Instruct/config.json")

INFO 12-18 05:27:03 [__init__.py:244] Automatically detected platform cuda.


In [5]:
from datasets import load_dataset as hf_load_dataset

# Load a large, diverse dataset for finding robust redundant directions
# Using Anthropic HH-RLHF which has diverse conversational data
print("Loading diverse dataset from Hugging Face...")
dataset = hf_load_dataset("Anthropic/hh-rlhf", split="train")

# Sample a large subset for diversity (10k examples)
import random
random.seed(42)
indices = random.sample(range(len(dataset)), min(10000, len(dataset)))
sampled_data = dataset.select(indices)

# Extract prompts (take the human turn from the conversation)
inputs = []
for item in sampled_data:
    # Parse the conversation format
    text = item['chosen']  # Use the chosen response path
    # Extract just the first human prompt
    if '\n\nHuman:' in text:
        parts = text.split('\n\nHuman:')
        if len(parts) > 1:
            prompt = parts[1].split('\n\nAssistant:')[0].strip()
            if prompt:
                inputs.append(prompt)

# Shuffle to ensure variety
random.shuffle(inputs)
inputs = inputs[:5000]  # Use 5k diverse prompts

print(f"Loaded {len(inputs)} diverse prompts")
print(f"\nSample prompts:")
for i in range(3):
    print(f"{i+1}. {inputs[i][:100]}...")


Loading diverse dataset from Hugging Face...
Loaded 5000 diverse prompts

Sample prompts:
1. Can you explain what DDOSing is and why people do it? It seems to happen a lot to video game servers...
2. Which is more important between limiting the spread of the virus, and getting people more healthy to...
3. I just learned about foreign currency exchange scams and it seems like a great thing to get into, th...
Loaded 5000 diverse prompts

Sample prompts:
1. Can you explain what DDOSing is and why people do it? It seems to happen a lot to video game servers...
2. Which is more important between limiting the spread of the virus, and getting people more healthy to...
3. I just learned about foreign currency exchange scams and it seems like a great thing to get into, th...


# Dataset: Diverse Conversational Data

We use **Anthropic HH-RLHF** dataset which contains:
- **Diverse topics**: Science, coding, philosophy, advice, creative writing, factual questions, etc.
- **Natural conversations**: Real human-AI interactions covering wide range of use cases
- **Large scale**: 5,000 sampled prompts ensuring statistical robustness

## Optimization Strategy for Diverse Data

With a diverse dataset, we adjust optimization parameters:

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `num_iterations` | 500 | More iterations needed for convergence across diverse gradients |
| `batch_size` | 8 | Balance between diversity and memory efficiency |
| `lr` | 0.01 | Lower learning rate for stability with diverse data |
| `projection_weight` | 0.1 | Balanced weight for redundancy signals |

This ensures the found redundant directions **generalize** across conversation types rather than being artifacts of a specific domain.


In [6]:
import pandas as pd
from src.data import TableLoader
from src.activ_extractor import ActivationExtractor


layer_name = f"model.layers.{refusal_config.layer_index}"

activ_extractor = ActivationExtractor(
    model,
    layer_name,
    capture_output=True,
)

In [7]:
# For this diverse dataset, we'll use prompts without targets
# This tests redundancy on the model's natural response distribution
# rather than being biased by specific target completions

# Create simple conversations (user prompts only, model will complete naturally)
convs = [[{"role": "user", "content": inp}] for inp in inputs]

print(f"Created {len(convs)} conversations")
print(f"Dataset diversity: {len(set(inputs))} unique prompts")


Created 5000 conversations
Dataset diversity: 4832 unique prompts


In [8]:
adv_model = AdvModel(
    model=model,
    tokenizer=tokenizer,
    num_tokens=0,
)

In [9]:
import matplotlib.pyplot as plt
import numpy as np

def plot_projection_norms(layer_activs, direction, tokens, tokenizer, num_prompts=10, normalize_activs=False, title_suffix=""):
    """
    Compute and plot projection norms of activations onto a direction vector.
    
    Args:
        layer_activs: Activations tensor (batch_size, seq_len, hidden_dim)
        direction: Direction vector (hidden_dim,)
        tokens: Tokenizer output with input_ids and attention_mask
        tokenizer: Tokenizer for decoding
        num_prompts: Number of prompts to visualize
        normalize_activs: Whether to L2 normalize activations before projection
        title_suffix: Suffix to add to plot titles
    """
    # Normalize activations if requested (add epsilon for numerical stability)
    if normalize_activs:
        layer_activs = layer_activs / (layer_activs.norm(dim=-1, keepdim=True) + 1e-8)
    
    # Normalize the direction vector to unit length
    direction_norm = direction / direction.norm()
    
    # Compute scalar projection: (x · v) where v is unit vector
    projection_scalars = layer_activs @ direction_norm  # shape: (batch_size, seq_len)
    
    # Compute L2 norm of the projections (absolute value of scalar projection)
    l2_norms = torch.abs(projection_scalars)  # shape: (batch_size, seq_len)
    
    # Move to CPU for visualization
    l2_norms_cpu = l2_norms.float().cpu().numpy()
    tokens_cpu = tokens.input_ids.cpu().numpy()
    attention_mask_cpu = tokens.attention_mask.cpu().numpy()
    
    # Get pad token ID
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    
    # Create visualization
    num_prompts = min(num_prompts, l2_norms_cpu.shape[0])
    fig, axes = plt.subplots(num_prompts, 1, figsize=(15, 3 * num_prompts))
    
    if num_prompts == 1:
        axes = [axes]
    
    for idx in range(num_prompts):
        ax = axes[idx]
        
        token_ids = tokens_cpu[idx]
        norms = l2_norms_cpu[idx]
        attention_mask = attention_mask_cpu[idx]
        
        # Filter out LEFT padding tokens only (attention_mask==0 AND token==pad_token)
        # This keeps assistant/generated tokens even though they have attention_mask==0
        is_left_padding = (attention_mask == 0) & (token_ids == pad_token_id)
        non_pad_mask = ~is_left_padding
        
        token_ids_filtered = token_ids[non_pad_mask]
        norms_filtered = norms[non_pad_mask]
        
        # Decode tokens
        token_strs = [tokenizer.decode([tid]) for tid in token_ids_filtered]
        
        # Create bar plot
        x_pos = np.arange(len(token_strs))
        ax.bar(x_pos, norms_filtered, alpha=0.7)
        
        # Set labels
        ax.set_xticks(x_pos)
        ax.set_xticklabels(token_strs, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("L2 Norm (Projection onto Direction)")
        ax.set_title(f"Prompt {idx + 1}: Token-wise Projection Magnitudes{title_suffix}")
        ax.grid(axis="y", alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\nProjection Statistics{title_suffix}:")
    print(f"Mean L2 norm: {l2_norms.mean().item():.4f}")
    print(f"Std L2 norm: {l2_norms.std().item():.4f}")
    print(f"Max L2 norm: {l2_norms.max().item():.4f}")
    print(f"Min L2 norm: {l2_norms.min().item():.4f}")



In [10]:
class ActivationManipulator(ActivationExtractor):
    """
    Extends ActivationExtractor to manipulate activations during forward pass.
    Manipulates the OUTPUT of the target layer (same as ActivationExtractor captures outputs).
    """
    
    def __init__(self, model, layer_name, manipulation_fn=None):
        """
        Args:
            model: PyTorch model
            layer_name: Name of the layer to manipulate
            manipulation_fn: Function that takes activations and returns modified activations
        """
        # Force capture_output=True to ensure we're working with outputs
        super().__init__(model, layer_name, exact_match=True, capture_output=True)
        self.manipulation_fn = manipulation_fn
        
    def set_manipulation(self, manipulation_fn):
        """Set the manipulation function to apply to activations."""
        self.manipulation_fn = manipulation_fn
        
    def _create_output_hook(self, layer_name: str):
        """Override to create a hook that manipulates the OUTPUT activations."""
        def hook_fn(module, args, output):
            # Handle tuple outputs (take first element)
            if isinstance(output, tuple):
                output_tensor = output[0]
                is_tuple = True
            else:
                output_tensor = output
                is_tuple = False
            
            # Apply manipulation if function is set
            if self.manipulation_fn is not None:
                modified_output = self.manipulation_fn(output_tensor)
                
                # Return in the same format as input
                if is_tuple:
                    return (modified_output,) + output[1:]
                else:
                    return modified_output
            
            return output
        
        return hook_fn
        
    def _register_hooks(self):
        """Register hooks that manipulate OUTPUT activations."""
        for layer_name, layer in self._layers:
            hook_fn = self._create_output_hook(layer_name)
            # Use register_forward_hook to manipulate outputs
            handle = layer.register_forward_hook(hook_fn, prepend=False, with_kwargs=False)
            self._handles.append(handle)

In [11]:
def compute_top_k_token_agreement(logits1, logits2, attention_mask, k=10):
    """
    Compute top-k token agreement between two sets of logits, over all non-padding tokens.
    
    This measures how often the top-k predicted tokens are the same between two distributions.
    A better metric than KL-div for our purpose since we care about which tokens are predicted,
    not the exact probability distribution.
    
    Args:
        logits1: First logits tensor (batch_size, seq_len, vocab_size)
        logits2: Second logits tensor (batch_size, seq_len, vocab_size)
        attention_mask: Attention mask (batch_size, seq_len) - 1 for real tokens, 0 for padding
        k: Number of top tokens to consider
        
    Returns:
        Average top-k agreement score (1.0 = perfect agreement, 0.0 = no overlap)
    """
    # Align predicted logits with input tokens (shift by 1)
    logits1 = logits1[:, :-1]  # remove last prediction
    logits2 = logits2[:, :-1]  # remove last prediction
    attention_mask = attention_mask[:, 1:]  # remove first token (we predict starting from second position)
    
    # Extract only non-padding tokens
    non_pad_mask = attention_mask.bool()
    non_pad_logits1 = logits1[non_pad_mask].view(-1, logits1.size(-1))
    non_pad_logits2 = logits2[non_pad_mask].view(-1, logits2.size(-1))
    
    # Get top-k tokens for each position
    top_k_1 = torch.topk(non_pad_logits1, k, dim=-1).indices  # (num_tokens, k)
    top_k_2 = torch.topk(non_pad_logits2, k, dim=-1).indices  # (num_tokens, k)
    
    # Compute overlap: how many of top-k tokens are the same
    # For each position, count how many tokens appear in both top-k lists
    agreements = []
    for i in range(top_k_1.shape[0]):
        overlap = len(set(top_k_1[i].tolist()) & set(top_k_2[i].tolist()))
        agreements.append(overlap / k)
    
    return torch.tensor(agreements).mean()


def compute_top_1_accuracy(logits1, logits2, attention_mask):
    """
    Compute top-1 token accuracy: how often is the most likely token the same?
    Over all non-padding tokens.
    
    Args:
        logits1: First logits tensor (batch_size, seq_len, vocab_size)
        logits2: Second logits tensor (batch_size, seq_len, vocab_size)
        attention_mask: Attention mask (batch_size, seq_len) - 1 for real tokens, 0 for padding
        
    Returns:
        Fraction of positions where top-1 token matches
    """
    # Align predicted logits with input tokens (shift by 1)
    logits1 = logits1[:, :-1]
    logits2 = logits2[:, :-1]
    attention_mask = attention_mask[:, 1:]
    
    # Extract only non-padding tokens
    non_pad_mask = attention_mask.bool()
    non_pad_logits1 = logits1[non_pad_mask].view(-1, logits1.size(-1))
    non_pad_logits2 = logits2[non_pad_mask].view(-1, logits2.size(-1))
    
    # Get top-1 tokens
    top1_1 = torch.argmax(non_pad_logits1, dim=-1)
    top1_2 = torch.argmax(non_pad_logits2, dim=-1)
    
    # Compute accuracy
    accuracy = (top1_1 == top1_2).float().mean()
    
    return accuracy


def compute_kl_divergence(logits1, logits2, attention_mask):
    """
    Compute KL divergence between two logit distributions over all non-padding tokens.
    
    Args:
        logits1: First logits tensor (batch_size, seq_len, vocab_size)
        logits2: Second logits tensor (batch_size, seq_len, vocab_size)
        attention_mask: Attention mask (batch_size, seq_len) - 1 for real tokens, 0 for padding
        
    Returns:
        Mean KL divergence across all non-padding positions
    """
    # Align predicted logits with input tokens (shift by 1)
    logits1 = logits1[:, :-1]
    logits2 = logits2[:, :-1]
    attention_mask = attention_mask[:, 1:]
    
    # Extract only non-padding tokens
    non_pad_mask = attention_mask.bool()
    non_pad_logits1 = logits1[non_pad_mask].view(-1, logits1.size(-1))
    non_pad_logits2 = logits2[non_pad_mask].view(-1, logits2.size(-1))
    
    # Convert to log probabilities
    log_probs1 = torch.nn.functional.log_softmax(non_pad_logits1, dim=-1)
    log_probs2 = torch.nn.functional.log_softmax(non_pad_logits2, dim=-1)
    
    # Compute KL divergence: KL(P1 || P2) = sum(P1 * (log P1 - log P2))
    probs1 = torch.exp(log_probs1)
    kl_div = (probs1 * (log_probs1 - log_probs2)).sum(dim=-1).mean()
    
    return kl_div


In [12]:
def find_redundant_1d_subspace(
    model,
    tokenizer,
    adv_model,
    layer_name,
    conversations,
    num_iterations=200,
    lr=0.01,
    projection_weight=0.1,
    batch_size=32,
    device="cuda"
):
    """
    Find a redundant 1D subspace: a direction where activations have large projections,
    but removing this projection doesn't significantly change the output tokens.
    
    This identifies linear subspaces that the model "uses" but that are somewhat redundant
    for the final prediction.
    
    T is a projection operator: T = v v^T where v is a unit vector (1D projection)
    For activation e, we subtract T(e) = (v^T e) v from e
    
    We want to find v such that:
    - Top-1 token predictions remain unchanged (high agreement) - PRIMARY OBJECTIVE
    - ||T(e)|| is large for NORMALIZED activations (proportional importance) - SECONDARY
    
    CRITICAL: v is constrained to be a unit vector (||v|| = 1) throughout optimization.
    This prevents trivially scaling up the projection by multiplying v by a large constant.
    We achieve this using reparametrization: optimize unconstrained w, then use v = w/||w||
    
    Args:
        model: The language model
        tokenizer: Tokenizer
        adv_model: AdvModel wrapper
        layer_name: Name of layer to manipulate
        conversations: List of ALL conversations (will be batched internally)
        num_iterations: Number of optimization steps
        lr: Learning rate
        projection_weight: Weight for projection term (default 0.1, lower = more emphasis on distribution preservation)
        batch_size: Number of conversations per batch
        device: Device to run on
        
    Returns:
        Optimized direction vector v (unit vector)
    """
    # Initialize UNCONSTRAINED direction vector
    hidden_dim = model.config.hidden_size
    direction_unconstrained = torch.randn(hidden_dim, device=device, dtype=torch.bfloat16)
    direction_unconstrained.requires_grad = True
    
    # Setup optimizer on the unconstrained vector
    optimizer = torch.optim.Adam([direction_unconstrained], lr=lr)
    
    # Setup activation manipulator
    manipulator = ActivationManipulator(model, layer_name)
    
    # Calculate number of batches
    num_conversations = len(conversations)
    num_batches = (num_conversations + batch_size - 1) // batch_size
    
    print(f"Searching for redundant 1D subspace over {num_iterations} iterations...")
    print(f"Using {num_conversations} conversations in {num_batches} batches of size {batch_size}")
    print("Goal: PRIMARY - preserve distribution when ablating projection")
    print("      SECONDARY - maximize proportional projection (on normalized activations)")
    print(f"Loss balance: KL-div weight=1.0, projection weight={projection_weight}")
    print("Constraint: Direction vector maintained at unit norm via reparametrization")
    print("Evaluation: Over all non-padding tokens\n")
    
    best_score = -float('inf')
    best_direction = None
    
    for iteration in range(num_iterations):
        # Sample a random batch of conversations for this iteration
        batch_indices = torch.randperm(num_conversations)[:batch_size].tolist()
        batch_convs = [conversations[i] for i in batch_indices]
        
        # Tokenize batch
        tokens = adv_model.tokenize(batch_convs).to(device)
        
        optimizer.zero_grad()
        
        # Normalize to get the actual unit direction vector
        # This reparametrization ensures ||v|| = 1 automatically
        direction = direction_unconstrained / direction_unconstrained.norm()
        
        # Get baseline logits (without manipulation)
        with torch.no_grad():
            baseline_logits = adv_model.forward(
                tokens.input_ids,
                tokens.attention_mask,
                adv_mask=None
            ).logits
        
        # Define manipulation function: e -> e - T(e) where T(e) = (v^T e) v
        def subtract_projection(activations):
            proj_scalars = activations @ direction
            projection = proj_scalars.unsqueeze(-1) * direction.unsqueeze(0).unsqueeze(0)
            return activations - projection
        
        manipulator.set_manipulation(subtract_projection)
        
        # Forward pass with manipulation
        with manipulator.capture():
            modified_logits = adv_model.forward(
                tokens.input_ids,
                tokens.attention_mask,
                adv_mask=None
            ).logits
        
        # Compute top-k agreement and top-1 accuracy (over all non-padding tokens)
        top10_agreement = compute_top_k_token_agreement(
            baseline_logits,
            modified_logits,
            tokens.attention_mask,
            k=10
        )
        
        top1_accuracy = compute_top_1_accuracy(
            baseline_logits,
            modified_logits,
            tokens.attention_mask
        )
        
        # Compute KL divergence (lower is better - distribution preserved)
        kl_div = compute_kl_divergence(
            baseline_logits,
            modified_logits,
            tokens.attention_mask
        )
        
        # Compute projection magnitude on NORMALIZED activations
        # This measures the proportional importance of the projection
        # CRITICAL: Use capture_output=True to get OUTPUT activations (same as manipulator)
        activ_extractor = ActivationExtractor(model, layer_name, capture_output=True)
        with activ_extractor.capture():
            _ = adv_model.forward(
                tokens.input_ids,
                tokens.attention_mask,
                adv_mask=None
            )
        activations = activ_extractor.get_activations()[layer_name]
        
        # Normalize activations to unit vectors
        # This makes us measure the PROPORTIONAL component, not absolute magnitude
        activations_normalized = activations / (activations.norm(dim=-1, keepdim=True) + 1e-8)
        
        # Compute mean absolute projection magnitude on normalized activations
        # Now bounded between 0 and 1 (since both activation and direction are unit vectors)
        proj_scalars = activations_normalized @ direction
        projection_norm = torch.abs(proj_scalars).mean()
        
        # Loss function with configurable balance between objectives
        # primary_loss: minimize KL divergence (preserve distribution)
        # secondary_loss: maximize proportional projection magnitude (find non-trivial directions)
        primary_loss = kl_div
        secondary_loss = -projection_norm  # Projection on normalized activations
        
        # Combined loss with configurable weight
        # projection_weight controls the relative importance of projection vs. distribution preservation
        loss = primary_loss + projection_weight * secondary_loss
        
        loss.backward()
        optimizer.step()
        
        # Note: No manual normalization needed! The reparametrization handles it.
        # The gradient flows through the normalization operation automatically.
        
        # Track best: prioritize low KL-div, then high projection as tiebreaker
        # Score now reflects: high accuracy + high projection, but accuracy is more important
        score = top1_accuracy.item() * 10.0 + projection_norm.item()
        if score > best_score:
            best_score = score
            with torch.no_grad():
                best_direction = (direction_unconstrained / direction_unconstrained.norm()).clone()
        
        if iteration % 20 == 0:
            print(f"Iter {iteration:3d}: Top-1-Acc={top1_accuracy.item():.4f}, "
                  f"KL-Div={kl_div.item():.6f}, "
                  f"Proj-Norm={projection_norm.item():.4f}, "
                  f"Loss={loss.item():.6f}, "
                  f"||v||={direction.norm().item():.6f}")
    
    # Return best direction found (already normalized by construction)
    if best_direction is None:
        with torch.no_grad():
            best_direction = direction_unconstrained / direction_unconstrained.norm()
    
    print(f"\n✓ Best score: {best_score:.4f}")
    print(f"✓ Final direction norm: {best_direction.norm().item():.6f}")
    return best_direction


In [ ]:
# Validate the learned redundant direction
def validate_redundant_subspace(model, adv_model, layer_name, tokens, direction, batch_size=8):
    """
    Validate the learned direction by measuring:
    1. How much activations project onto it (magnitude)
    2. How little removing it affects the output (redundancy)
    
    Computes metrics over all non-padding tokens.
    Uses batched processing for memory efficiency.
    
    Args:
        model: The language model
        adv_model: AdvModel wrapper
        layer_name: Name of layer to validate
        tokens: Tokenized conversations (full dataset)
        direction: Direction vector to validate
        batch_size: Number of samples to process per batch
        
    Returns:
        Dictionary of validation metrics
    """
    # Process in batches to avoid memory issues
    num_samples = tokens.input_ids.shape[0]
    num_batches = (num_samples + batch_size - 1) // batch_size
    
    all_top1_accuracies = []
    all_top10_agreements = []
    all_kl_divs = []
    all_proj_scalars_raw = []
    all_proj_scalars_normalized = []
    
    print(f"Validating over {num_samples} samples in {num_batches} batches...")
    
    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, num_samples)
        
        # Extract batch
        batch_input_ids = tokens.input_ids[start_idx:end_idx]
        batch_attention_mask = tokens.attention_mask[start_idx:end_idx]
        
        # Get baseline logits
        with torch.no_grad():
            baseline_logits = adv_model.forward(
                batch_input_ids,
                batch_attention_mask,
                adv_mask=None
            ).logits
        
        # Setup manipulator
        manipulator = ActivationManipulator(model, layer_name)
        
        def subtract_projection(activations):
            proj_scalars = activations @ direction
            projection = proj_scalars.unsqueeze(-1) * direction.unsqueeze(0).unsqueeze(0)
            return activations - projection
        
        manipulator.set_manipulation(subtract_projection)
        
        # Get modified logits
        with torch.no_grad():
            with manipulator.capture():
                modified_logits = adv_model.forward(
                    batch_input_ids,
                    batch_attention_mask,
                    adv_mask=None
                ).logits
        
        # Compute metrics over all non-padding tokens in this batch
        top1_accuracy = compute_top_1_accuracy(
            baseline_logits,
            modified_logits,
            batch_attention_mask
        )
        all_top1_accuracies.append(top1_accuracy.item())
        
        top10_agreement = compute_top_k_token_agreement(
            baseline_logits,
            modified_logits,
            batch_attention_mask,
            k=10
        )
        all_top10_agreements.append(top10_agreement.item())
        
        kl_div = compute_kl_divergence(
            baseline_logits,
            modified_logits,
            batch_attention_mask
        )
        all_kl_divs.append(kl_div.item())
        
        # Compute projection statistics on RAW activations
        # CRITICAL: Use capture_output=True to get OUTPUT activations (same as manipulator)
        activ_extractor = ActivationExtractor(model, layer_name, capture_output=True)
        with activ_extractor.capture():
            _ = adv_model.forward(
                batch_input_ids,
                batch_attention_mask,
                adv_mask=None
            )
        activations = activ_extractor.get_activations()[layer_name]
        
        # Raw projection (absolute magnitude)
        proj_scalars_raw = activations @ direction
        all_proj_scalars_raw.append(torch.abs(proj_scalars_raw))
        
        # Normalized projection (proportional importance, 0-1 range)
        activations_normalized = activations / (activations.norm(dim=-1, keepdim=True) + 1e-8)
        proj_scalars_normalized = activations_normalized @ direction
        all_proj_scalars_normalized.append(torch.abs(proj_scalars_normalized))
        
        if (batch_idx + 1) % 5 == 0 or (batch_idx + 1) == num_batches:
            print(f"  Processed {end_idx}/{num_samples} samples...")
    
    # Aggregate results across all batches
    # For metrics, compute weighted average (though all batches should contribute equally)
    avg_top1_accuracy = sum(all_top1_accuracies) / len(all_top1_accuracies)
    avg_top10_agreement = sum(all_top10_agreements) / len(all_top10_agreements)
    avg_kl_div = sum(all_kl_divs) / len(all_kl_divs)
    
    # For projections, concatenate all tensors and compute statistics
    all_proj_raw = torch.cat(all_proj_scalars_raw, dim=0)
    all_proj_normalized = torch.cat(all_proj_scalars_normalized, dim=0)
    
    projection_mean_raw = all_proj_raw.mean().item()
    projection_std_raw = all_proj_raw.std().item()
    projection_mean_normalized = all_proj_normalized.mean().item()
    projection_std_normalized = all_proj_normalized.std().item()
    
    return {
        'top1_accuracy': avg_top1_accuracy,
        'top10_agreement': avg_top10_agreement,
        'kl_divergence': avg_kl_div,
        'projection_mean_raw': projection_mean_raw,
        'projection_std_raw': projection_std_raw,
        'projection_mean_normalized': projection_mean_normalized,
        'projection_std_normalized': projection_std_normalized,
        'redundancy_score': projection_mean_raw * avg_top1_accuracy
    }

# Validate on a random sample of conversations
num_val_samples = 50
val_indices = torch.randperm(len(convs))[:num_val_samples].tolist()
val_convs = [convs[i] for i in val_indices]
validation_tokens = adv_model.tokenize(val_convs).to(adv_model.device)


# Multi-Location Redundant Subspace Search

We'll conduct a comprehensive exploration of redundant 1D projections across the model architecture:

## 1. **Residual Stream (Main Information Highway)**
   - **Early** (layers 3, 7): Initial processing after embeddings
   - **Middle** (layers 14, 18): Core semantic transformations
   - **Late** (layers 24, 27): Final refinement before prediction
   - Most likely to accumulate redundancy across depth

## 2. **MLP Component Outputs**
   - **After gate_proj** (layers 10, 20): 3072→8192 expansion, pre-activation
   - **After up_proj** (layers 10, 20): 3072→8192 expansion, parallel path
   - **After act_fn (SiLU)** (layers 10, 20): Post-activation, pre-gating
   - **After down_proj** (layers 10, 15, 20, 25): 8192→3072 bottleneck (high redundancy potential)

## 3. **Attention Component Outputs**
   - **After q_proj** (layers 12, 22): Query projections before attention
   - **After k_proj** (layers 12, 22): Key projections (lower dim: 1024)
   - **After v_proj** (layers 12, 22): Value projections (lower dim: 1024)
   - **After o_proj** (layers 12, 18, 22): Final attention output before residual

## 4. **Normalization Outputs**
   - **After input_layernorm** (layers 5, 15, 25): Pre-attention normalization
   - **After post_attention_layernorm** (layers 5, 15, 25): Pre-MLP normalization

## 5. **Special Locations**
   - **After embed_tokens**: Raw embeddings before any processing
   - **After final norm**: Last residual stream before lm_head

This comprehensive search will reveal where redundancy emerges architecturally.

In [ ]:
# Define comprehensive candidate locations to search
# Format: (location_name, layer_path, description)

search_locations = [
    # ========== RESIDUAL STREAM (Main Information Highway) ==========
    ("residual_very_early", "model.layers.3", "Very early residual (layer 3, post-embedding processing)"),
    ("residual_early", "model.layers.7", "Early residual (layer 7, initial semantic processing)"),
    ("residual_middle", "model.layers.14", "Middle residual (layer 14, core transformations)"),
    ("residual_mid_late", "model.layers.18", "Mid-late residual (layer 18, refined semantics)"),
    ("residual_late", "model.layers.24", "Late residual (layer 24, pre-output refinement)"),
    ("residual_final", "model.layers.27", "Final residual (layer 27, last processing stage)"),
    
    # ========== MLP INTERNALS (Feed-Forward Processing) ==========
    # Gate projection path (3072 → 8192)
    ("mlp_gate_mid", "model.layers.10.mlp.gate_proj", "MLP gate projection middle (layer 10, pre-activation)"),
    ("mlp_gate_late", "model.layers.20.mlp.gate_proj", "MLP gate projection late (layer 20, pre-activation)"),
    
    # Up projection path (3072 → 8192, parallel to gate)
    ("mlp_up_mid", "model.layers.10.mlp.up_proj", "MLP up projection middle (layer 10, parallel path)"),
    ("mlp_up_late", "model.layers.20.mlp.up_proj", "MLP up projection late (layer 20, parallel path)"),
    
    # Down projection (8192 → 3072 BOTTLENECK - High redundancy potential)
    ("mlp_down_early", "model.layers.10.mlp.down_proj", "MLP down proj early (layer 10, bottleneck)"),
    ("mlp_down_mid", "model.layers.15.mlp.down_proj", "MLP down proj middle (layer 15, bottleneck)"),
    ("mlp_down_late", "model.layers.20.mlp.down_proj", "MLP down proj late (layer 20, bottleneck)"),
    ("mlp_down_very_late", "model.layers.25.mlp.down_proj", "MLP down proj very late (layer 25, bottleneck)"),
    
    # Complete MLP output (after all processing)
    ("mlp_output_mid", "model.layers.15.mlp", "Complete MLP output middle (layer 15)"),
    ("mlp_output_late", "model.layers.20.mlp", "Complete MLP output late (layer 20)"),
    
    # ========== ATTENTION COMPONENTS ==========
    # Query projections (3072 → 3072)
    ("attn_q_mid", "model.layers.12.self_attn.q_proj", "Attention Q projection middle (layer 12)"),
    ("attn_q_late", "model.layers.22.self_attn.q_proj", "Attention Q projection late (layer 22)"),
    
    # Key projections (3072 → 1024, dimensional reduction)
    ("attn_k_mid", "model.layers.12.self_attn.k_proj", "Attention K projection middle (layer 12, dim=1024)"),
    ("attn_k_late", "model.layers.22.self_attn.k_proj", "Attention K projection late (layer 22, dim=1024)"),
    
    # Value projections (3072 → 1024, dimensional reduction)
    ("attn_v_mid", "model.layers.12.self_attn.v_proj", "Attention V projection middle (layer 12, dim=1024)"),
    ("attn_v_late", "model.layers.22.self_attn.v_proj", "Attention V projection late (layer 22, dim=1024)"),
    
    # Output projections (3072 → 3072, post-attention)
    ("attn_o_mid", "model.layers.12.self_attn.o_proj", "Attention O projection middle (layer 12)"),
    ("attn_o_mid_late", "model.layers.18.self_attn.o_proj", "Attention O projection mid-late (layer 18)"),
    ("attn_o_late", "model.layers.22.self_attn.o_proj", "Attention O projection late (layer 22)"),
    
    # Complete attention output
    ("attn_output_mid", "model.layers.12.self_attn", "Complete attention output middle (layer 12)"),
    ("attn_output_late", "model.layers.22.self_attn", "Complete attention output late (layer 22)"),
    
    # ========== NORMALIZATION OUTPUTS ==========
    # Input LayerNorm (pre-attention)
    ("norm_input_early", "model.layers.5.input_layernorm", "Input norm early (layer 5, pre-attention)"),
    ("norm_input_mid", "model.layers.15.input_layernorm", "Input norm middle (layer 15, pre-attention)"),
    ("norm_input_late", "model.layers.25.input_layernorm", "Input norm late (layer 25, pre-attention)"),
    
    # Post-Attention LayerNorm (pre-MLP)
    ("norm_post_attn_early", "model.layers.5.post_attention_layernorm", "Post-attn norm early (layer 5, pre-MLP)"),
    ("norm_post_attn_mid", "model.layers.15.post_attention_layernorm", "Post-attn norm middle (layer 15, pre-MLP)"),
    ("norm_post_attn_late", "model.layers.25.post_attention_layernorm", "Post-attn norm late (layer 25, pre-MLP)"),
    
    # ========== SPECIAL LOCATIONS ==========
    ("embeddings", "model.embed_tokens", "Token embeddings (raw, before any layers)"),
    ("final_norm", "model.norm", "Final normalization (last residual, before lm_head)"),
]

print("=" * 80)
print(f"COMPREHENSIVE REDUNDANCY EXPLORATION: {len(search_locations)} LOCATIONS")
print("=" * 80)
print(f"\nWill search for redundant projections across the model architecture:")
print(f"  • 6 Residual stream locations (depth progression)")
print(f"  • 10 MLP component locations (gate, up, down projections + outputs)")
print(f"  • 13 Attention component locations (q, k, v, o projections + outputs)")
print(f"  • 6 Normalization output locations (pre-attn, pre-MLP)")
print(f"  • 2 Special locations (embeddings, final norm)")
print(f"\n{'Name':<25} {'Path':<50} Description")
print("=" * 80)
for name, path, desc in search_locations:
    print(f"  {name:<25} {path:<50} {desc}")

Will search for redundant projections at:
  residual_early       → model.layers.7                 | Early residual stream (after layer 7)
  residual_middle      → model.layers.14                | Middle residual stream (after layer 14)
  residual_late        → model.layers.24                | Late residual stream (after layer 24)
  mlp_middle           → model.layers.15.mlp            | MLP output middle (layer 15, after down_proj)
  mlp_late             → model.layers.20.mlp            | MLP output late (layer 20, after down_proj)
  attn_late            → model.layers.22.self_attn      | Attention output late (layer 22, after o_proj)


In [ ]:
# Multi-location search: Find redundant directions at each location
import time

# Storage for results
location_results = {}

# Batch size for both optimization and validation
BATCH_SIZE = 8

print("=" * 80)
print("SEARCHING FOR REDUNDANT 1D SUBSPACES ACROSS MULTIPLE LOCATIONS")
print("=" * 80)
print(f"Using {len(convs)} diverse prompts from Anthropic HH-RLHF dataset")
print(f"Batch size: {BATCH_SIZE} (used for both optimization and validation)")
print("This ensures redundant directions generalize across diverse conversation types\n")

for location_name, layer_path, description in search_locations:
    print(f"\n{'=' * 80}")
    print(f"LOCATION: {location_name}")
    print(f"Path: {layer_path}")
    print(f"Description: {description}")
    print(f"{'=' * 80}\n")
    
    start_time = time.time()
    
    # Find redundant direction at this location
    # Adjusted parameters for larger, more diverse dataset:
    # - More iterations (500) for better convergence on diverse data
    # - Batch size (8) for memory efficiency
    # - Lower learning rate (0.01) for stability with diverse gradients
    # - projection_weight (0.1) for balanced redundancy signals
    redundant_dir = find_redundant_1d_subspace(
        model=model,
        tokenizer=tokenizer,
        adv_model=adv_model,
        layer_name=layer_path,
        conversations=convs,
        num_iterations=500,  # More iterations for diverse data
        lr=0.01,  # Lower LR for stability
        projection_weight=0.1,  # Balanced weight
        batch_size=BATCH_SIZE,
        device="cuda"
    )
    
    # Validate on a separate sample using same batch size
    num_val_samples = 100  # More validation samples for diverse dataset
    val_indices = torch.randperm(len(convs))[:num_val_samples].tolist()
    val_convs = [convs[i] for i in val_indices]
    val_tokens = adv_model.tokenize(val_convs).to(adv_model.device)
    
    print(f"\nValidating on {num_val_samples} samples...")
    val_metrics = validate_redundant_subspace(
        model, adv_model, layer_path, val_tokens, redundant_dir, batch_size=BATCH_SIZE
    )
    
    elapsed = time.time() - start_time
    
    # Store results
    location_results[location_name] = {
        'layer_path': layer_path,
        'description': description,
        'direction': redundant_dir,
        'metrics': val_metrics,
        'time_sec': elapsed
    }
    
    print(f"\n✓ Completed in {elapsed:.1f}s")
    print(f"  Redundancy Score:           {val_metrics['redundancy_score']:.4f}")
    print(f"  Top-1 Accuracy:             {val_metrics['top1_accuracy']:.4f}")
    print(f"  KL Divergence:              {val_metrics['kl_divergence']:.6f}")
    print(f"  Raw Projection Mean:        {val_metrics['projection_mean_raw']:.4f}")
    print(f"  Normalized Projection Mean: {val_metrics['projection_mean_normalized']:.4f}  ⭐")

print(f"\n{'=' * 80}")
print("ALL LOCATIONS SEARCHED")
print(f"{'=' * 80}")


SEARCHING FOR REDUNDANT 1D SUBSPACES ACROSS MULTIPLE LOCATIONS
Using 5000 diverse prompts from Anthropic HH-RLHF dataset
This ensures redundant directions generalize across diverse conversation types


LOCATION: residual_early
Path: model.layers.7
Description: Early residual stream (after layer 7)

Searching for redundant 1D subspace over 500 iterations...
Using 5000 conversations in 625 batches of size 8
Goal: PRIMARY - preserve distribution when ablating projection
      SECONDARY - maximize proportional projection (on normalized activations)
Loss balance: KL-div weight=1.0, projection weight=0.1
Constraint: Direction vector maintained at unit norm via reparametrization
Evaluation: Over all non-padding tokens

Iter   0: Top-1-Acc=0.9337, KL-Div=0.011230, Proj-Norm=0.0150, Loss=0.009766, ||v||=1.000000
Iter   0: Top-1-Acc=0.9337, KL-Div=0.011230, Proj-Norm=0.0150, Loss=0.009766, ||v||=1.000000
Iter  20: Top-1-Acc=0.9922, KL-Div=0.002701, Proj-Norm=0.0203, Loss=0.000671, ||v||=1.000000

OutOfMemoryError: CUDA out of memory. Tried to allocate 12.19 GiB. GPU 0 has a total capacity of 47.51 GiB of which 8.97 GiB is free. Including non-PyTorch memory, this process has 38.53 GiB memory in use. Of the allocated memory 36.69 GiB is allocated by PyTorch, and 1.33 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Compare results across all locations
import pandas as pd

print("\n" + "=" * 100)
print("COMPARISON: REDUNDANT SUBSPACES ACROSS LOCATIONS")
print("=" * 100 + "\n")

# Create comparison dataframe
comparison_data = []
for location_name, result in location_results.items():
    metrics = result['metrics']
    comparison_data.append({
        'Location': location_name,
        'Description': result['description'],
        'Redundancy Score': f"{metrics['redundancy_score']:.4f}",
        'Top-1 Accuracy': f"{metrics['top1_accuracy']:.4f}",
        'KL Divergence': f"{metrics['kl_divergence']:.6f}",
        'Proj. Mean (Raw)': f"{metrics['projection_mean_raw']:.4f}",
        'Proj. Mean (Norm)': f"{metrics['projection_mean_normalized']:.4f}",
        'Proj. Std (Raw)': f"{metrics['projection_std_raw']:.4f}",
        'Time (s)': f"{result['time_sec']:.1f}"
    })

df_comparison = pd.DataFrame(comparison_data)

# Sort by redundancy score (descending)
df_comparison = df_comparison.sort_values('Redundancy Score', ascending=False)

print(df_comparison.to_string(index=False))
print("\nKey: Proj. Mean (Norm) = Normalized/Proportional projection (0-1 range) ⭐")

print("\n" + "=" * 80)
print("KEY INSIGHTS")
print("=" * 80)

# Find best location by redundancy score
best_location = max(location_results.items(), key=lambda x: x[1]['metrics']['redundancy_score'])
print(f"\n1. STRONGEST REDUNDANCY:")
print(f"   Location: {best_location[0]}")
print(f"   {best_location[1]['description']}")
print(f"   Redundancy Score: {best_location[1]['metrics']['redundancy_score']:.4f}")
print(f"   Top-1 Accuracy: {best_location[1]['metrics']['top1_accuracy']:.4f}")
print(f"   Normalized Proj. Mean: {best_location[1]['metrics']['projection_mean_normalized']:.4f} ({best_location[1]['metrics']['projection_mean_normalized']*100:.1f}%)")

# Find location with best distribution preservation (lowest KL-div)
best_kl = min(location_results.items(), key=lambda x: x[1]['metrics']['kl_divergence'])
print(f"\n2. BEST DISTRIBUTION PRESERVATION:")
print(f"   Location: {best_kl[0]}")
print(f"   {best_kl[1]['description']}")
print(f"   KL Divergence: {best_kl[1]['metrics']['kl_divergence']:.6f}")

# Find location with largest NORMALIZED projections (most important)
best_proj_norm = max(location_results.items(), key=lambda x: x[1]['metrics']['projection_mean_normalized'])
print(f"\n3. LARGEST PROPORTIONAL PROJECTIONS:")
print(f"   Location: {best_proj_norm[0]}")
print(f"   {best_proj_norm[1]['description']}")
print(f"   Normalized Proj. Mean: {best_proj_norm[1]['metrics']['projection_mean_normalized']:.4f} ({best_proj_norm[1]['metrics']['projection_mean_normalized']*100:.1f}%)")
print(f"   Raw Proj. Mean: {best_proj_norm[1]['metrics']['projection_mean_raw']:.4f}")

print("\n" + "=" * 80)
print("RECOMMENDATIONS")
print("=" * 80)
print("\nFor further analysis, focus on locations with:")
print("  • Top-1 Accuracy > 0.90 (minimal output change)")
print("  • Normalized Proj. Mean > 0.10 (>10% proportional importance)")
print("  • KL Divergence < 0.01 (strong distribution preservation)")
print("\nThese locations contain redundant subspaces that could be safely ablated.")

In [ ]:
# Visualize projections for the top 3 locations by redundancy score

# Get top 3 locations
sorted_locations = sorted(
    location_results.items(), 
    key=lambda x: x[1]['metrics']['redundancy_score'], 
    reverse=True
)[:3]

print("\n" + "=" * 80)
print("VISUALIZATION: TOP 3 LOCATIONS BY REDUNDANCY")
print("=" * 80)

for rank, (location_name, result) in enumerate(sorted_locations, 1):
    print(f"\n{'-' * 80}")
    print(f"RANK {rank}: {location_name}")
    print(f"{result['description']}")
    print(f"Redundancy Score: {result['metrics']['redundancy_score']:.4f}")
    print(f"{'-' * 80}")
    
    layer_path = result['layer_path']
    redundant_dir = result['direction']
    
    # Get activations at this location
    activ_extractor = ActivationExtractor(model, layer_path, capture_output=True)
    
    # Use validation tokens from earlier
    num_val_samples = 50
    val_indices = torch.randperm(len(convs))[:num_val_samples].tolist()
    val_convs = [convs[i] for i in val_indices]
    vis_tokens = adv_model.tokenize(val_convs).to(adv_model.device)
    
    with activ_extractor.capture():
        _ = adv_model.forward(
            vis_tokens.input_ids,
            vis_tokens.attention_mask,
            adv_mask=None
        )
    activations = activ_extractor.get_activations()[layer_path]
    
    # Plot normalized projections (proportional importance)
    print("\n--- Normalized Activation Projections (Proportional) ---")
    plot_projection_norms(
        activations, 
        redundant_dir, 
        vis_tokens, 
        tokenizer, 
        num_prompts=3,  # Show fewer prompts for cleaner comparison
        normalize_activs=True, 
        title_suffix=f" [{location_name}]"
    )

# Long-Term Generation Test

Now we'll test whether the redundant directions have **long-term dependencies** on generation quality.

Our optimization focused on **next-token KL divergence**, but we need to verify that ablating these directions doesn't accumulate errors over multi-token generation.

We'll generate full responses with and without ablation for the top 3 locations and compare:
- **Text similarity** (exact match, edit distance)
- **Semantic coherence** (length, structure)
- **Qualitative differences** (side-by-side comparison)

This tests whether the redundancy is truly **local** (next-token only) or if there are **cascading effects** in autoregressive generation.

In [ ]:
def generate_with_ablation(
    model,
    tokenizer,
    adv_model,
    layer_name,
    direction,
    prompts,
    max_new_tokens=100,
    ablate=True
):
    """
    Generate text with optional direction ablation at a specific layer.
    
    Args:
        model: Language model
        tokenizer: Tokenizer
        adv_model: AdvModel wrapper
        layer_name: Layer to ablate at
        direction: Direction vector to ablate
        prompts: List of prompt strings
        max_new_tokens: Maximum tokens to generate
        ablate: Whether to ablate the direction (False = baseline)
        
    Returns:
        List of generated texts
    """
    # Prepare conversations
    convs = [[{"role": "user", "content": prompt}] for prompt in prompts]
    
    # Setup manipulator if ablating
    manipulator = None
    if ablate:
        manipulator = ActivationManipulator(model, layer_name)
        
        def subtract_projection(activations):
            proj_scalars = activations @ direction
            projection = proj_scalars.unsqueeze(-1) * direction.unsqueeze(0).unsqueeze(0)
            return activations - projection
        
        manipulator.set_manipulation(subtract_projection)
    
    # Generate
    generated_texts = []
    
    for conv in convs:
        # Tokenize prompt only (no target)
        tokens = adv_model.tokenize([conv]).to(adv_model.device)
        input_ids = tokens.input_ids
        attention_mask = tokens.attention_mask
        
        # Generate with or without ablation
        if ablate:
            with manipulator.capture():
                outputs = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,  # Greedy decoding for reproducibility
                    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                )
        else:
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )
        
        # Decode (skip prompt tokens)
        generated_ids = outputs[0][input_ids.shape[1]:]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        generated_texts.append(generated_text)
    
    return generated_texts


def compute_text_similarity(text1, text2):
    """
    Compute various similarity metrics between two texts.
    
    Returns dict with:
        - exact_match: Whether texts are identical
        - length_diff: Difference in character length
        - edit_distance: Levenshtein distance (normalized)
        - token_overlap: Fraction of tokens that match
    """
    import difflib
    
    exact_match = (text1 == text2)
    length_diff = abs(len(text1) - len(text2))
    
    # Normalized edit distance (0 = identical, 1 = completely different)
    max_len = max(len(text1), len(text2), 1)
    edit_distance = difflib.SequenceMatcher(None, text1, text2).ratio()
    
    # Token-level overlap
    tokens1 = text1.split()
    tokens2 = text2.split()
    if len(tokens1) == 0 and len(tokens2) == 0:
        token_overlap = 1.0
    elif len(tokens1) == 0 or len(tokens2) == 0:
        token_overlap = 0.0
    else:
        common = sum(1 for t1, t2 in zip(tokens1, tokens2) if t1 == t2)
        token_overlap = common / max(len(tokens1), len(tokens2))
    
    return {
        'exact_match': exact_match,
        'length_diff': length_diff,
        'similarity_ratio': edit_distance,  # Higher is more similar (0-1)
        'token_overlap': token_overlap
    }

In [ ]:
# Select test prompts - use diverse examples from different topics
# Ensure we test across various conversation types
num_test_prompts = 10  # More test cases for diverse dataset
test_prompt_indices = torch.randperm(len(inputs))[:num_test_prompts].tolist()
test_prompts = [inputs[i] for i in test_prompt_indices]

print("=" * 80)
print("TEST PROMPTS FOR LONG-TERM GENERATION")
print("=" * 80)
print("Selected diverse prompts to test redundancy across different topics:\n")
for i, prompt in enumerate(test_prompts, 1):
    print(f"{i}. {prompt[:100]}..." if len(prompt) > 100 else f"{i}. {prompt}")


In [ ]:
# Test the top 3 locations with best redundancy scores
print("\n" + "=" * 80)
print("LONG-TERM GENERATION TEST: TOP 3 REDUNDANT LOCATIONS")
print("=" * 80)
print("\nGenerating responses with and without ablation...")
print("This tests if redundancy is LOCAL (next-token) or has CASCADING effects.\n")

generation_results = {}

for rank, (location_name, result) in enumerate(sorted_locations, 1):
    print(f"\n{'=' * 80}")
    print(f"TESTING LOCATION {rank}: {location_name}")
    print(f"Description: {result['description']}")
    print(f"{'=' * 80}\n")
    
    layer_path = result['layer_path']
    redundant_dir = result['direction']
    
    # Generate baseline (no ablation)
    print("Generating baseline outputs (NO ablation)...")
    baseline_outputs = generate_with_ablation(
        model=model,
        tokenizer=tokenizer,
        adv_model=adv_model,
        layer_name=layer_path,
        direction=redundant_dir,
        prompts=test_prompts,
        max_new_tokens=100,
        ablate=False
    )
    
    # Generate with ablation
    print("Generating ablated outputs (WITH ablation)...")
    ablated_outputs = generate_with_ablation(
        model=model,
        tokenizer=tokenizer,
        adv_model=adv_model,
        layer_name=layer_path,
        direction=redundant_dir,
        prompts=test_prompts,
        max_new_tokens=100,
        ablate=True
    )
    
    # Compute similarities
    similarities = []
    for baseline, ablated in zip(baseline_outputs, ablated_outputs):
        sim = compute_text_similarity(baseline, ablated)
        similarities.append(sim)
    
    # Aggregate statistics
    exact_matches = sum(s['exact_match'] for s in similarities)
    avg_similarity = sum(s['similarity_ratio'] for s in similarities) / len(similarities)
    avg_token_overlap = sum(s['token_overlap'] for s in similarities) / len(similarities)
    avg_length_diff = sum(s['length_diff'] for s in similarities) / len(similarities)
    
    generation_results[location_name] = {
        'baseline_outputs': baseline_outputs,
        'ablated_outputs': ablated_outputs,
        'similarities': similarities,
        'stats': {
            'exact_matches': exact_matches,
            'avg_similarity': avg_similarity,
            'avg_token_overlap': avg_token_overlap,
            'avg_length_diff': avg_length_diff
        }
    }
    
    print(f"\n✓ Generation complete for {location_name}")
    print(f"  Exact matches: {exact_matches}/{len(test_prompts)}")
    print(f"  Avg similarity ratio: {avg_similarity:.4f} (1.0 = identical)")
    print(f"  Avg token overlap: {avg_token_overlap:.4f}")
    print(f"  Avg length diff: {avg_length_diff:.1f} characters")

print(f"\n{'=' * 80}")
print("GENERATION TESTING COMPLETE")
print(f"{'=' * 80}")

In [ ]:
# Summary comparison across all 3 locations
print("\n" + "=" * 80)
print("SUMMARY: LONG-TERM GENERATION IMPACT")
print("=" * 80 + "\n")

summary_data = []
for location_name, result in generation_results.items():
    stats = result['stats']
    summary_data.append({
        'Location': location_name,
        'Exact Matches': f"{stats['exact_matches']}/{len(test_prompts)}",
        'Avg Similarity': f"{stats['avg_similarity']:.4f}",
        'Avg Token Overlap': f"{stats['avg_token_overlap']:.4f}",
        'Avg Length Diff': f"{stats['avg_length_diff']:.1f}"
    })

df_generation = pd.DataFrame(summary_data)
print(df_generation.to_string(index=False))

print("\n" + "=" * 80)
print("INTERPRETATION")
print("=" * 80)

# Determine if redundancy is truly local
best_gen_location = max(generation_results.items(), key=lambda x: x[1]['stats']['avg_similarity'])
worst_gen_location = min(generation_results.items(), key=lambda x: x[1]['stats']['avg_similarity'])

print(f"\n✓ Best Preserved (Most Local Redundancy):")
print(f"  Location: {best_gen_location[0]}")
print(f"  Similarity: {best_gen_location[1]['stats']['avg_similarity']:.4f}")

print(f"\n✓ Worst Preserved (Potential Cascading Effects):")
print(f"  Location: {worst_gen_location[0]}")
print(f"  Similarity: {worst_gen_location[1]['stats']['avg_similarity']:.4f}")

print("\n" + "=" * 80)
print("CONCLUSION")
print("=" * 80)

# Overall assessment
overall_avg_sim = sum(r['stats']['avg_similarity'] for r in generation_results.values()) / len(generation_results)

if overall_avg_sim > 0.95:
    print("\n🎯 EXCELLENT: Redundancy is HIGHLY LOCAL")
    print(f"   Average similarity: {overall_avg_sim:.4f}")
    print("   → Ablating these directions has minimal long-term generation impact")
    print("   → Next-token KL-div optimization successfully captured the full effect")
elif overall_avg_sim > 0.85:
    print("\n✓ GOOD: Redundancy is MOSTLY LOCAL with minor cascading effects")
    print(f"   Average similarity: {overall_avg_sim:.4f}")
    print("   → Some minor differences accumulate during generation")
    print("   → Still safe to ablate, but monitor for edge cases")
else:
    print("\n⚠ CAUTION: Significant cascading effects detected")
    print(f"   Average similarity: {overall_avg_sim:.4f}")
    print("   → Ablation has substantial long-term generation impact")
    print("   → May not be truly redundant for autoregressive generation")

In [ ]:
# Detailed side-by-side comparison for qualitative analysis
print("\n" + "=" * 100)
print("DETAILED SIDE-BY-SIDE COMPARISON")
print("=" * 100)

for location_name, result in generation_results.items():
    print(f"\n{'=' * 100}")
    print(f"LOCATION: {location_name}")
    print(f"{'=' * 100}\n")
    
    baseline_outputs = result['baseline_outputs']
    ablated_outputs = result['ablated_outputs']
    similarities = result['similarities']
    
    for i, (prompt, baseline, ablated, sim) in enumerate(zip(test_prompts, baseline_outputs, ablated_outputs, similarities), 1):
        print(f"\n{'-' * 100}")
        print(f"EXAMPLE {i}")
        print(f"{'-' * 100}")
        print(f"Prompt: {prompt[:150]}..." if len(prompt) > 150 else f"Prompt: {prompt}")
        print(f"\nSimilarity: {sim['similarity_ratio']:.4f} | Token Overlap: {sim['token_overlap']:.4f} | Exact Match: {sim['exact_match']}")
        print(f"\n{'[BASELINE - No Ablation]':^100}")
        print(f"{'-' * 100}")
        print(baseline)
        print(f"\n{'[ABLATED - With Direction Removed]':^100}")
        print(f"{'-' * 100}")
        print(ablated)
        
        if not sim['exact_match']:
            print(f"\n{'[DIFFERENCE DETECTED]':^100}")
            # Show first difference location
            for j, (c1, c2) in enumerate(zip(baseline, ablated)):
                if c1 != c2:
                    context_start = max(0, j-30)
                    context_end = min(len(baseline), j+30)
                    print(f"First difference at position {j}:")
                    print(f"  Baseline: ...{baseline[context_start:context_end]}...")
                    print(f"  Ablated:  ...{ablated[context_start:context_end]}...")
                    break

print(f"\n{'=' * 100}")
print("END OF DETAILED COMPARISON")
print(f"{'=' * 100}")

In [ ]:
# Visualize similarity distribution across prompts and locations
import matplotlib.pyplot as plt
import numpy as np

print("\n" + "=" * 80)
print("VISUALIZATION: GENERATION SIMILARITY DISTRIBUTION")
print("=" * 80)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (location_name, result) in enumerate(generation_results.items()):
    ax = axes[idx]
    similarities = result['similarities']
    
    # Extract metrics
    similarity_ratios = [s['similarity_ratio'] for s in similarities]
    token_overlaps = [s['token_overlap'] for s in similarities]
    
    # Plot
    x_pos = np.arange(len(test_prompts))
    width = 0.35
    
    ax.bar(x_pos - width/2, similarity_ratios, width, label='Character Similarity', alpha=0.8, color='blue')
    ax.bar(x_pos + width/2, token_overlaps, width, label='Token Overlap', alpha=0.8, color='green')
    
    # Formatting
    ax.set_xlabel('Test Prompt #')
    ax.set_ylabel('Similarity Score (0-1)')
    ax.set_title(f'{location_name}\nAvg Sim: {result["stats"]["avg_similarity"]:.3f}')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'{i+1}' for i in range(len(test_prompts))])
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1.05])
    
    # Add horizontal line at 0.95 (high similarity threshold)
    ax.axhline(y=0.95, color='red', linestyle='--', alpha=0.5, label='High Similarity (0.95)')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  • Bars near 1.0 indicate generation is nearly identical")
print("  • Bars below 0.95 (red line) suggest accumulating differences")
print("  • Character similarity vs token overlap shows lexical vs semantic preservation")

# Normalization Layer Invariance Test

We need to verify that the redundant directions we found are **not simply artifacts of normalization layer invariance**.

**Key Insight:** LayerNorm and RMSNorm are invariant to certain directions:
- **LayerNorm**: Invariant to the constant direction `v = [1, 1, 1, ..., 1]` (mean-centering removes it)
- **RMSNorm**: Invariant to directions in the span of the input (scaling removes radial components)

If our redundant directions are simply aligned with these normalization-invariant directions, they would be **trivially redundant** rather than genuinely interesting.

**Tests:**
1. **Cosine similarity** with constant vector [1,1,1,...,1]
2. **Alignment analysis** across all found directions
3. **Architectural context**: Check if normalization layers exist after our target layers

In [ ]:
# First, let's understand the normalization architecture after each target layer
print("=" * 80)
print("NORMALIZATION LAYER ANALYSIS")
print("=" * 80)

# Check what comes after each target layer
for location_name, result in location_results.items():
    layer_path = result['layer_path']
    print(f"\n{location_name} ({layer_path}):")
    
    # Parse the layer path to understand the architecture
    if "model.layers." in layer_path:
        parts = layer_path.split(".")
        layer_idx = int(parts[2])
        
        if ".mlp" in layer_path:
            print(f"  → Target: MLP output (after down_proj)")
            print(f"  → Next: Residual addition, then post_attention_layernorm")
            print(f"  → Normalization: RMSNorm (input_layernorm for next layer)")
        elif ".self_attn" in layer_path:
            print(f"  → Target: Attention output (after o_proj)")
            print(f"  → Next: Residual addition, then post_attention_layernorm")
            print(f"  → Normalization: RMSNorm (post_attention_layernorm)")
        else:
            print(f"  → Target: Full decoder layer output")
            print(f"  → Next: input_layernorm for next layer")
            print(f"  → Normalization: RMSNorm (input_layernorm of layer {layer_idx + 1})")

print(f"\n{'=' * 80}")
print("CONCLUSION: All locations are followed by RMSNorm")
print("=" * 80)
print("\nRMSNorm formula: y = (x / sqrt(mean(x²))) * scale")
print("Invariance: RMSNorm is NOT invariant to constant directions [1,1,1,...]")
print("           (unlike LayerNorm which removes mean)")
print("\nHowever, we should still check alignment to rule out artifacts.")

In [ ]:
# Compute cosine similarity with constant vector and other invariant directions
print("\n" + "=" * 80)
print("TESTING ALIGNMENT WITH NORMALIZATION-INVARIANT DIRECTIONS")
print("=" * 80)

def compute_direction_alignments(direction, hidden_dim):
    """
    Compute alignment of a direction with various potentially invariant directions.
    
    Returns dict with cosine similarities.
    """
    direction_normalized = direction / direction.norm()
    
    # 1. Constant vector [1, 1, 1, ..., 1] - LayerNorm invariant
    constant_vec = torch.ones(hidden_dim, device=direction.device, dtype=direction.dtype)
    constant_vec_normalized = constant_vec / constant_vec.norm()
    cosine_constant = torch.abs(direction_normalized @ constant_vec_normalized).item()
    
    # 2. Random baseline (for comparison)
    random_vec = torch.randn(hidden_dim, device=direction.device, dtype=direction.dtype)
    random_vec_normalized = random_vec / random_vec.norm()
    cosine_random = torch.abs(direction_normalized @ random_vec_normalized).item()
    
    # 3. Standard basis vectors (check if direction is axis-aligned)
    max_component = torch.abs(direction_normalized).max().item()
    
    # 4. Check sparsity (how many components are significant)
    threshold = 0.1
    num_significant = (torch.abs(direction_normalized) > threshold).sum().item()
    sparsity_ratio = num_significant / hidden_dim
    
    return {
        'cosine_constant': cosine_constant,
        'cosine_random': cosine_random,
        'max_component': max_component,
        'num_significant_components': num_significant,
        'sparsity_ratio': sparsity_ratio
    }


# Analyze all found redundant directions
hidden_dim = model.config.hidden_size
alignment_results = {}

print(f"\nHidden dimension: {hidden_dim}")
print(f"Expected random cosine similarity: ~{1/np.sqrt(hidden_dim):.6f}")
print(f"\n{'-' * 80}")

for location_name, result in location_results.items():
    direction = result['direction']
    
    alignments = compute_direction_alignments(direction, hidden_dim)
    alignment_results[location_name] = alignments
    
    print(f"\n{location_name}:")
    print(f"  Cosine with constant [1,1,1,...]: {alignments['cosine_constant']:.6f}")
    print(f"  Cosine with random vector:        {alignments['cosine_random']:.6f}")
    print(f"  Max single component:             {alignments['max_component']:.6f}")
    print(f"  Significant components (>0.1):    {alignments['num_significant_components']}/{hidden_dim} ({alignments['sparsity_ratio']:.2%})")
    
    # Interpretation
    if alignments['cosine_constant'] > 0.5:
        print(f"  ⚠️  HIGH alignment with constant vector - may be normalization artifact!")
    elif alignments['cosine_constant'] > 0.2:
        print(f"  ⚠️  MODERATE alignment with constant vector - check carefully")
    else:
        print(f"  ✓ LOW alignment with constant vector - not a normalization artifact")

print(f"\n{'-' * 80}")

In [ ]:
# Create a summary table
print("\n" + "=" * 80)
print("SUMMARY: ALIGNMENT ANALYSIS")
print("=" * 80 + "\n")

alignment_data = []
for location_name, alignments in alignment_results.items():
    alignment_data.append({
        'Location': location_name,
        'Cos(constant)': f"{alignments['cosine_constant']:.6f}",
        'Cos(random)': f"{alignments['cosine_random']:.6f}",
        'Max Component': f"{alignments['max_component']:.6f}",
        'Sparsity': f"{alignments['sparsity_ratio']:.2%}",
        'Significant': f"{alignments['num_significant_components']}"
    })

df_alignments = pd.DataFrame(alignment_data)
print(df_alignments.to_string(index=False))

print("\n" + "=" * 80)
print("STATISTICAL ANALYSIS")
print("=" * 80)

# Compare with expected random baseline
expected_random = 1 / np.sqrt(hidden_dim)
avg_cosine_constant = np.mean([a['cosine_constant'] for a in alignment_results.values()])
avg_cosine_random = np.mean([a['cosine_random'] for a in alignment_results.values()])

print(f"\nExpected random alignment: {expected_random:.6f}")
print(f"Observed avg constant alignment: {avg_cosine_constant:.6f}")
print(f"Observed avg random alignment: {avg_cosine_random:.6f}")

# Statistical test: are our directions more aligned with constant than random?
constant_alignments = [a['cosine_constant'] for a in alignment_results.values()]
random_alignments = [a['cosine_random'] for a in alignment_results.values()]

from scipy import stats
if len(constant_alignments) >= 3:
    t_stat, p_value = stats.ttest_rel(constant_alignments, random_alignments)
    print(f"\nPaired t-test (constant vs random):")
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value: {p_value:.6f}")
    
    if p_value < 0.05 and np.mean(constant_alignments) > np.mean(random_alignments):
        print(f"  ⚠️ SIGNIFICANT: Directions ARE more aligned with constant vector!")
        print(f"     This suggests normalization artifacts may be involved.")
    else:
        print(f"  ✓ NOT SIGNIFICANT: Directions are not preferentially aligned with constant.")
        print(f"     Redundancy is likely NOT due to normalization invariance.")

print("\n" + "=" * 80)
print("INTERPRETATION GUIDELINES")
print("=" * 80)
print("\nCosine similarity thresholds:")
print("  • < 0.1: Essentially orthogonal (expected for random in high-D)")
print("  • 0.1-0.3: Weak alignment (normal variation)")
print("  • 0.3-0.5: Moderate alignment (potential artifact)")
print("  • > 0.5: Strong alignment (likely normalization artifact)")
print("\nFor reference:")
print(f"  • Hidden dim: {hidden_dim}")
print(f"  • Expected random: ~{expected_random:.6f}")
print(f"  • 2σ above random: ~{expected_random * 3:.6f}")

In [ ]:
# Visualize the direction components to understand their structure
print("\n" + "=" * 80)
print("VISUALIZING DIRECTION STRUCTURE")
print("=" * 80)

fig, axes = plt.subplots(len(location_results), 1, figsize=(15, 4 * len(location_results)))

if len(location_results) == 1:
    axes = [axes]

for idx, (location_name, result) in enumerate(location_results.items()):
    ax = axes[idx]
    direction = result['direction'].cpu().float().numpy()
    
    # Sort by absolute magnitude
    sorted_indices = np.argsort(np.abs(direction))[::-1]
    top_n = 100  # Show top 100 components
    
    x_pos = np.arange(top_n)
    ax.bar(x_pos, direction[sorted_indices[:top_n]], alpha=0.7)
    
    ax.set_xlabel('Component Rank (by magnitude)')
    ax.set_ylabel('Component Value')
    ax.set_title(f'{location_name}: Top {top_n} Direction Components\n' + 
                 f'Constant alignment: {alignment_results[location_name]["cosine_constant"]:.4f}')
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.grid(axis='y', alpha=0.3)
    
    # Add text annotations
    max_val = np.max(np.abs(direction))
    mean_val = np.mean(np.abs(direction))
    ax.text(0.02, 0.98, f'Max: {max_val:.4f}\nMean: {mean_val:.6f}', 
            transform=ax.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  • Uniform/flat distribution → may resemble constant vector")
print("  • Sparse/peaked distribution → structured, not normalization artifact")
print("  • Random-looking distribution → genuine redundant subspace")